# MedGemma-4B fit + stability test (Kaggle / Colab)
Goal: find out, with evidence, which compute dtype trains MedGemma-4B without NaN on this GPU.
**Settings (Kaggle):** Accelerator = GPU T4 x2 (or P100), Internet = ON.
**Token:** add your Hugging Face token as a Kaggle Secret named `HF_TOKEN` (Add-ons -> Secrets). Never paste it into a cell.
Run cells top to bottom and send the final report cell output back.

In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate pillow

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = os.environ.get("HF_TOKEN")
assert token, "Add HF_TOKEN as a Kaggle/Colab secret first"
from huggingface_hub import login
login(token=token)
print("logged in")

In [ ]:
import torch, subprocess
assert torch.cuda.is_available(), "Turn on a GPU accelerator"
n = torch.cuda.device_count()
for i in range(n):
    p = torch.cuda.get_device_properties(i)
    print(i, p.name, round(p.total_memory/1024**3,1), "GB, compute", p.major, p.minor)
cc = torch.cuda.get_device_capability(0)[0]
print("native bf16:", cc >= 8, "| torch bf16 flag:", torch.cuda.is_bf16_supported())
# order of dtypes to try: bf16 only if the GPU is Ampere or newer
DTYPES = ["bfloat16"] if cc >= 8 else ["float16", "float32"]
print("will try, in order:", DTYPES)

In [ ]:
import gc, time
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "google/medgemma-4b-it"
processor = AutoProcessor.from_pretrained(MODEL_ID)

def run(dtype_name, steps=3):
    dt = getattr(torch, dtype_name)
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dt)
    t0 = time.time()
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID, quantization_config=bnb, device_map={"": 0}, torch_dtype=dt)
    load_s = time.time() - t0
    out = {"dtype": dtype_name, "load_s": round(load_s), "vram_after_load_gb": round(torch.cuda.memory_allocated()/1024**3, 2)}

    # 1) inference sanity
    img = Image.new("RGB", (448, 448), (128, 128, 128))
    msgs = [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": "Is a fracture visible? Answer yes or no."}]}]
    inp = processor.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt").to(0)
    with torch.inference_mode():
        gen = model.generate(**inp, max_new_tokens=20, do_sample=False)
    out["sample_answer"] = processor.decode(gen[0][inp["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

    # 2) tiny LoRA training run
    model.gradient_checkpointing_enable()
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model = get_peft_model(model, LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, task_type="CAUSAL_LM",
                           target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]))
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=1e-4)
    full = msgs + [{"role": "assistant", "content": [{"type": "text", "text": "No, a fracture is not visible."}]}]
    batch = processor.apply_chat_template(full, tokenize=True, return_dict=True, return_tensors="pt").to(0)
    batch["labels"] = batch["input_ids"].clone()
    model.train()
    losses = []
    for _ in range(steps):
        loss = model(**batch).loss
        losses.append(float(loss))
        loss.backward(); opt.step(); opt.zero_grad()
    out["losses"] = losses
    out["finite"] = all(l == l and abs(l) != float("inf") for l in losses)
    out["peak_vram_gb"] = round(torch.cuda.max_memory_allocated()/1024**3, 2)
    del model, opt; gc.collect(); torch.cuda.empty_cache()
    return out

results = []
for d in DTYPES:
    try:
        r = run(d)
    except Exception as e:
        r = {"dtype": d, "error": f"{type(e).__name__}: {str(e)[:300]}"}
    print(r); results.append(r)
    if r.get("finite"):
        break

In [ ]:
print("===== REPORT: copy this back =====")
for r in results: print(r)
good = [r for r in results if r.get("finite")]
print("RECOMMENDED DTYPE:", good[0]["dtype"] if good else "NONE WORKED - need a bf16 GPU (L4/A100) or other fix")